# Day 1：Scaled Dot-Product Attention 从原理到代码

> 学习主线：今天只攻克一件事——**Attention 到底如何让一个 token 读取另一个 token 的信息**。  
> 最低产出：你必须能手写 `QKᵀ / √dₖ / softmax / V`，能画出矩阵维度，能解释为什么要除以 `√dₖ`，能回答面试追问。

---

## 今日学习内容大纲

| 模块 | 你要学会什么 | 产出 |
|---|---|---|
| 0. 今日目标 | 明确 Attention 的学习边界 | 一张学习路线表 |
| 1. Q / K / V 直觉 | Query 是“我要找什么”，Key 是“我能被怎么找到”，Value 是“我携带什么内容” | 能口述 Q/K/V 角色 |
| 2. 矩阵维度流 | 从 `X` 到 `Q/K/V`，再到 `[T,T]` 注意力矩阵 | 能手写维度推导 |
| 3. Scaled Dot-Product Attention | `softmax(QKᵀ/√dₖ)V` 的完整计算 | NumPy 复现 |
| 4. 为什么除以 `√dₖ` | 点积方差随维度增大，softmax 会饱和 | 可视化 softmax 饱和 |
| 5. 面试拷问 | 为什么不是 `QQᵀ` / `KKᵀ`？为什么不用普通 dot product？ | 标准回答模板 |
| 6. 工程映射 | ChatModel / PromptTemplate / Runnable 最小链路 | 理解“模型调用链”和底层 attention 的边界 |
| 7. 今日作业 | 手写公式 + 改代码 + 解释排障 | 自测清单 |

---

## 完成目标

学完这个 Notebook 后，你应该能做到：

1. 看到 `Q: [B,T,d_k]`、`K: [B,T,d_k]`、`V: [B,T,d_v]` 时不再害怕维度。
2. 解释为什么 `QKᵀ` 是所有 token 两两之间的相关性表。
3. 用 NumPy 从零实现 scaled dot-product attention。
4. 通过图表说明：`d_k` 变大时，如果不除以 `√d_k`，softmax 会变尖、梯度会变小。
5. 面试时能把 Attention 公式讲成工程直觉，而不是只背公式。

## 0. 环境准备

这个 Notebook 只依赖：

- `numpy`
- `matplotlib`
- `pandas`

可选：

- `torch`：最后有一个 PyTorch 对照实现，没装也不影响主线学习。

建议运行环境：

- 本地 Jupyter / VS Code Notebook
- Google Colab
- Kaggle Notebook

先执行下面的环境单元。

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 尽量让中文图表标题和坐标轴正常显示。
# 若本地没有 Noto Sans CJK，Matplotlib 会自动回退到可用字体。
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
for font_name in ["Noto Sans CJK SC", "Noto Sans CJK JP", "SimHei", "Microsoft YaHei", "Arial Unicode MS"]:
    if font_name in available_fonts:
        plt.rcParams["font.sans-serif"] = [font_name]
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=3, suppress=True)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib font:", plt.rcParams.get("font.sans-serif", ["default"])[0])

# 1. Attention 的核心公式

Transformer 中的 Scaled Dot-Product Attention 定义为：

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

其中：

- $Q$：Query，查询向量，表示“我当前 token 想找什么信息”
- $K$：Key，键向量，表示“我这个 token 能被怎样匹配”
- $V$：Value，值向量，表示“我真正携带的内容”
- $d_k$：每个 key/query 向量的维度
- $QK^\top$：两两匹配分数矩阵
- $\sqrt{d_k}$：缩放因子，避免 logits 过大导致 softmax 饱和

---

## 矩阵维度

假设输入 embedding 是：

$$
X \in \mathbb{R}^{B \times T \times d_{model}}
$$

三个线性投影：

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V
$$

对应维度：

$$
W_Q \in \mathbb{R}^{d_{model}\times d_k},\quad
W_K \in \mathbb{R}^{d_{model}\times d_k},\quad
W_V \in \mathbb{R}^{d_{model}\times d_v}
$$

因此：

$$
Q \in \mathbb{R}^{B\times T\times d_k},\quad
K \in \mathbb{R}^{B\times T\times d_k},\quad
V \in \mathbb{R}^{B\times T\times d_v}
$$

注意力分数：

$$
QK^\top: [B,T,d_k]\times[B,d_k,T]\rightarrow[B,T,T]
$$

最终输出：

$$
A V: [B,T,T]\times[B,T,d_v]\rightarrow[B,T,d_v]
$$

一句话：**每个 token 用自己的 Query 去匹配所有 token 的 Key，再用匹配权重加权所有 token 的 Value。**

In [ ]:
# 用表格明确今天最核心的维度流
B = 2
T = 5
d_model = 8
d_k = 4
d_v = 6

shape_table = pd.DataFrame([
    ["X", "输入 token embeddings", f"[B,T,d_model] = [{B},{T},{d_model}]"],
    ["W_Q", "Query 投影矩阵", f"[d_model,d_k] = [{d_model},{d_k}]"],
    ["W_K", "Key 投影矩阵", f"[d_model,d_k] = [{d_model},{d_k}]"],
    ["W_V", "Value 投影矩阵", f"[d_model,d_v] = [{d_model},{d_v}]"],
    ["Q = XW_Q", "查询向量", f"[B,T,d_k] = [{B},{T},{d_k}]"],
    ["K = XW_K", "键向量", f"[B,T,d_k] = [{B},{T},{d_k}]"],
    ["V = XW_V", "值向量", f"[B,T,d_v] = [{B},{T},{d_v}]"],
    ["QKᵀ", "两两相关性分数", f"[B,T,T] = [{B},{T},{T}]"],
    ["softmax(QKᵀ/√d_k)", "每行归一化后的注意力权重", f"[B,T,T] = [{B},{T},{T}]"],
    ["Attention output", "加权汇聚后的上下文表示", f"[B,T,d_v] = [{B},{T},{d_v}]"],
], columns=["符号", "含义", "维度"])

shape_table

# 2. 先画一张 Attention 数据流图

这张图是今天最重要的“脑内白板”。  
只要你能把这张图画出来，Attention 的维度主线就已经通了。

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_box(ax, xy, text, width=2.4, height=0.75):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.03",
        linewidth=1.5,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(x + width/2, y + height/2, text, ha="center", va="center", fontsize=10)

def draw_arrow(ax, start, end, label=None):
    arrow = FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=14, linewidth=1.4)
    ax.add_patch(arrow)
    if label:
        mx = (start[0] + end[0]) / 2
        my = (start[1] + end[1]) / 2
        ax.text(mx, my + 0.12, label, ha="center", fontsize=9)

fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(0, 13)
ax.set_ylim(0, 6)
ax.axis("off")

draw_box(ax, (0.4, 2.6), "X\n[B,T,d_model]")
draw_box(ax, (3.0, 4.4), "Q = XW_Q\n[B,T,d_k]")
draw_box(ax, (3.0, 2.6), "K = XW_K\n[B,T,d_k]")
draw_box(ax, (3.0, 0.8), "V = XW_V\n[B,T,d_v]")
draw_box(ax, (6.0, 3.5), "Scores = QKᵀ\n[B,T,T]")
draw_box(ax, (8.7, 3.5), "Scale\n/√d_k")
draw_box(ax, (8.7, 2.0), "Softmax row-wise\nA: [B,T,T]")
draw_box(ax, (11.2, 2.0), "O = A V\n[B,T,d_v]")

draw_arrow(ax, (2.8, 3.0), (3.0, 4.75), "linear")
draw_arrow(ax, (2.8, 3.0), (3.0, 2.95), "linear")
draw_arrow(ax, (2.8, 3.0), (3.0, 1.15), "linear")

draw_arrow(ax, (5.4, 4.75), (6.0, 3.95), "Q")
draw_arrow(ax, (5.4, 2.95), (6.0, 3.85), "Kᵀ")
draw_arrow(ax, (8.4, 3.9), (8.7, 3.9))
draw_arrow(ax, (9.9, 3.5), (9.9, 2.8))
draw_arrow(ax, (11.0, 2.4), (11.2, 2.4), "× V")
draw_arrow(ax, (5.4, 1.15), (11.2, 2.05), "Value 被权重加权汇聚")

ax.set_title("Scaled Dot-Product Attention 数据流：从 X 到 Attention Output", fontsize=15, pad=16)
plt.show()

# 3. NumPy 从零实现 Scaled Dot-Product Attention

下面的函数就是今天必须手写出来的最小实现。

核心步骤：

```text
1. scores = Q @ K.T
2. scaled_scores = scores / sqrt(d_k)
3. weights = softmax(scaled_scores)
4. output = weights @ V
```

注意：softmax 必须按最后一个维度做，也就是每个 query token 对所有 key token 的权重之和为 1。

In [ ]:
def softmax(x, axis=-1):
    # 数值稳定版 softmax。
    # 减去 max 不改变 softmax 结果，但能避免 exp 溢出。
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, return_intermediates=True):
    # Q: [B, T, d_k]
    # K: [B, T, d_k]
    # V: [B, T, d_v]
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2)          # [B,T,T]
    scaled_scores = scores / math.sqrt(d_k)     # [B,T,T]
    weights = softmax(scaled_scores, axis=-1)   # [B,T,T]
    output = weights @ V                        # [B,T,d_v]

    if return_intermediates:
        return output, scores, scaled_scores, weights
    return output

In [ ]:
# 构造一个玩具输入，观察完整形状
rng = np.random.default_rng(42)

B, T, d_model, d_k, d_v = 1, 5, 8, 4, 6
X = rng.normal(size=(B, T, d_model))
W_Q = rng.normal(size=(d_model, d_k))
W_K = rng.normal(size=(d_model, d_k))
W_V = rng.normal(size=(d_model, d_v))

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

O, scores, scaled_scores, weights = scaled_dot_product_attention(Q, K, V)

print("X.shape       =", X.shape)
print("Q.shape       =", Q.shape)
print("K.shape       =", K.shape)
print("V.shape       =", V.shape)
print("scores.shape  =", scores.shape)
print("weights.shape =", weights.shape)
print("O.shape       =", O.shape)
print()
print("每个 query token 对所有 key token 的 attention 权重和：")
print(weights.sum(axis=-1))

# 4. 可视化：`QKᵀ` 分数矩阵与 softmax 后的注意力权重

下面我们用 5 个 token 做一个小例子：

```text
["The", "cat", "sat", "near", "fire"]
```

`QKᵀ` 的第 `i` 行第 `j` 列表示：

> 第 `i` 个 token 的 Query 对第 `j` 个 token 的 Key 的匹配程度。

softmax 后，每一行会变成概率分布，表示：

> 第 `i` 个 token 应该从哪些 token 的 Value 里读信息。

In [ ]:
tokens = ["The", "cat", "sat", "near", "fire"]

def plot_matrix(matrix, title, tokens, fmt="{:.2f}"):
    fig, ax = plt.subplots(figsize=(7.2, 5.8))
    im = ax.imshow(matrix)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=30, ha="right")
    ax.set_yticklabels(tokens)
    ax.set_xlabel("Key / 被关注的 token")
    ax.set_ylabel("Query / 当前正在更新的 token")
    ax.set_title(title, pad=14)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, fmt.format(matrix[i, j]), ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

plot_matrix(scores[0], "Raw scores = QKᵀ：还没有缩放、还没有归一化", tokens)
plot_matrix(weights[0], "Attention weights = softmax(QKᵀ/√d_k)：每一行和为 1", tokens)

# 5. 对单个 token 进行“逐项拆账”

假设我们看第 2 个 token：`cat`。  
它的输出不是凭空生成的，而是：

$$
O_{cat} = \sum_j A_{cat,j} V_j
$$

也就是：`cat` 根据自己的注意力权重，从所有 token 的 value 中加权读信息。

In [ ]:
query_index = 1  # cat
df_weights = pd.DataFrame({
    "Key token": tokens,
    "Attention weight from 'cat'": weights[0, query_index]
}).sort_values("Attention weight from 'cat'", ascending=False)

df_weights

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(df_weights["Key token"], df_weights["Attention weight from 'cat'"])
ax.set_title("Token 'cat' 从各个 token 读取 Value 的权重")
ax.set_xlabel("Key / Value token")
ax.set_ylabel("Attention weight")
ax.set_ylim(0, max(df_weights["Attention weight from 'cat'"]) * 1.25)

for idx, value in enumerate(df_weights["Attention weight from 'cat'"]):
    ax.text(idx, value, f"{value:.3f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

# 6. 为什么点积能代表相关性？

两个向量的点积：

$$
a\cdot b = \|a\|\|b\|\cos\theta
$$

如果两个向量方向接近，夹角 $\theta$ 小，$\cos\theta$ 大，点积就大。  
如果两个向量方向接近正交，点积接近 0。  
如果方向相反，点积为负。

在 Attention 中：

- Query：当前 token 想找的信息方向
- Key：候选 token 提供的信息标签方向

所以 `QKᵀ` 的含义就是：

> “我想找的方向”和“你提供的索引方向”是否匹配？

In [ ]:
# 画几个二维向量，直观看点积与方向关系
vectors = {
    "q": np.array([1.0, 0.6]),
    "k_similar": np.array([0.9, 0.5]),
    "k_orthogonal": np.array([-0.6, 1.0]),
    "k_opposite": np.array([-1.0, -0.5]),
}

q = vectors["q"]
rows = []
for name, v in vectors.items():
    if name == "q":
        continue
    dot = float(q @ v)
    cos = float(dot / (np.linalg.norm(q) * np.linalg.norm(v)))
    rows.append([name, dot, cos])

dot_df = pd.DataFrame(rows, columns=["Key vector", "q · k", "cosine similarity"])
dot_df

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

for name, v in vectors.items():
    ax.arrow(0, 0, v[0], v[1], head_width=0.04, length_includes_head=True)
    ax.text(v[0]*1.08, v[1]*1.08, name, fontsize=10)

ax.set_title("点积直觉：方向越匹配，Q·K 越大")
plt.show()

# 7. 为什么要除以 `√d_k`？

现在进入今天最容易被问到的核心问题。

假设 `q` 和 `k` 的每个维度都是均值 0、方差 1 的随机变量：

$$
q \cdot k = \sum_{i=1}^{d_k} q_i k_i
$$

如果每一项方差大约为 1，那么总和的方差大约会随 $d_k$ 增大：

$$
\mathrm{Var}(q\cdot k)\approx d_k
$$

也就是说，维度越高，点积的尺度越大。  
如果直接送进 softmax，softmax 会越来越尖，接近 one-hot。

除以 $\sqrt{d_k}$ 后：

$$
\mathrm{Var}\left(\frac{q\cdot k}{\sqrt{d_k}}\right)\approx 1
$$

这相当于把不同维度下的 attention logits 拉回稳定尺度。

In [ ]:
# 模拟不同 d_k 下 dot product 的标准差：不缩放 vs 除以 sqrt(d_k)
rng = np.random.default_rng(0)

dks = [4, 8, 16, 32, 64, 128, 256, 512]
n_samples = 20000

rows = []
for dk in dks:
    q = rng.normal(size=(n_samples, dk))
    k = rng.normal(size=(n_samples, dk))
    dot = np.sum(q * k, axis=1)
    scaled_dot = dot / math.sqrt(dk)
    rows.append([dk, dot.std(), scaled_dot.std()])

var_df = pd.DataFrame(rows, columns=["d_k", "std(q·k)", "std((q·k)/√d_k)"])
var_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(var_df["d_k"], var_df["std(q·k)"], marker="o", label="不缩放：std(q·k)")
ax.plot(var_df["d_k"], var_df["std((q·k)/√d_k)"], marker="o", label="缩放后：std((q·k)/√d_k)")
ax.set_xscale("log", base=2)
ax.set_xticks(dks)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel("head dimension d_k")
ax.set_ylabel("logits 标准差")
ax.set_title("为什么要除以 √d_k：不缩放时 logits 尺度随维度变大")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Softmax 饱和：为什么 logits 过大会导致梯度变小？

softmax 的输出如果接近 one-hot，例如：

```text
[0.999, 0.0005, 0.0005]
```

模型几乎只关注一个位置。  
这会带来两个问题：

1. 注意力分布过早极端，训练不稳定。
2. 大部分位置的梯度非常小，模型很难从其他 token 学到信息。

下面我们对比一组 logits 放大前后的 softmax。

In [ ]:
example_logits_small = np.array([1.2, 1.5, 1.8])
example_logits_big = example_logits_small * 10

sm_small = softmax(example_logits_small)
sm_big = softmax(example_logits_big)

softmax_compare = pd.DataFrame({
    "token": ["token_1", "token_2", "token_3"],
    "logits_small": example_logits_small,
    "softmax_small": sm_small,
    "logits_big": example_logits_big,
    "softmax_big": sm_big,
})

softmax_compare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

axes[0].bar(["token_1", "token_2", "token_3"], sm_small)
axes[0].set_title("logits 较小：softmax 较平滑")
axes[0].set_ylim(0, 1)
for i, v in enumerate(sm_small):
    axes[0].text(i, v, f"{v:.3f}", ha="center", va="bottom")

axes[1].bar(["token_1", "token_2", "token_3"], sm_big)
axes[1].set_title("logits 放大 10 倍：softmax 近似 one-hot")
for i, v in enumerate(sm_big):
    axes[1].text(i, v, f"{v:.3f}", ha="center", va="bottom")

fig.suptitle("Softmax 饱和现象：输入尺度过大，概率分布会变得极端", y=1.05)
plt.tight_layout()
plt.show()

## 梯度直觉：二分类 softmax / sigmoid 的导数

二分类时可以用 sigmoid 的导数做直觉类比：

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

$$
\sigma'(x)=\sigma(x)(1-\sigma(x))
$$

当 $x$ 很大或很小时，$\sigma(x)$ 接近 1 或 0，导数接近 0。  
这就是“饱和区域梯度小”的直观来源。

In [ ]:
xs = np.linspace(-12, 12, 400)
sigmoid = 1 / (1 + np.exp(-xs))
grad = sigmoid * (1 - sigmoid)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(xs, sigmoid, label="sigmoid(x)")
ax.plot(xs, grad, label="sigmoid'(x) = sigmoid(x)(1-sigmoid(x))")
ax.axvline(0, linewidth=1, linestyle="--")
ax.set_title("饱和区梯度变小：logits 过大时学习信号变弱")
ax.set_xlabel("logit x")
ax.set_ylabel("value")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 9. Masked Self-Attention：生成式模型为什么不能看未来？

今天主线是普通 self-attention，但大模型自回归生成时还会加 **causal mask**。

对第 `i` 个位置来说，它只能看见：

```text
0, 1, 2, ..., i
```

不能看见未来 token：

```text
i+1, i+2, ...
```

否则训练时就泄漏答案了。

数学上就是把未来位置的 score 设为一个极小值：

$$
-\infty
$$

softmax 后这些位置权重就会变成 0。

In [ ]:
def causal_mask(T):
    # 上三角未来位置为 True
    return np.triu(np.ones((T, T), dtype=bool), k=1)

T = 6
mask = causal_mask(T)
dummy_scores = rng.normal(size=(T, T))
masked_scores = dummy_scores.copy()
masked_scores[mask] = -1e9
masked_weights = softmax(masked_scores, axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(mask)
axes[0].set_title("Causal mask：未来位置被遮住")
axes[0].set_xlabel("Key position")
axes[0].set_ylabel("Query position")

axes[1].imshow(masked_weights)
axes[1].set_title("Masked attention weights：未来权重为 0")
axes[1].set_xlabel("Key position")
axes[1].set_ylabel("Query position")

for ax in axes:
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))

plt.tight_layout()
plt.show()

# 10. 面试官高频拷问

## 拷问 1：为什么是 `QKᵀ`，不是 `QQᵀ` 或 `KKᵀ`？

标准回答：

> Q 和 K 承担的是非对称角色。Q 是当前 token 的查询意图，K 是所有 token 可被匹配的索引标签。`QKᵀ` 表示“每个 token 想找什么”和“所有 token 能提供什么索引”之间的匹配关系。`QQᵀ` 只能表示 query 之间相似，`KKᵀ` 只能表示 key 之间相似，它们不能表达查询意图与索引标签之间的匹配。

进一步追问：

> Self-attention 里 Q/K/V 都来自同一个 X，为什么还要分三套矩阵？

回答：

> 因为同一个 token 在 attention 中要扮演不同角色：作为“查询者”、作为“被查询的索引”、作为“被读取的信息内容”。三套投影矩阵让模型在不同子空间里学习这三种角色。如果强行共用，会降低表达能力。

---

## 拷问 2：为什么要除以 `√d_k`？

标准回答：

> 不缩放时，`q·k` 的方差会随 `d_k` 增大，softmax 输入会变得很大，注意力分布过早接近 one-hot，导致梯度变小、训练不稳定。除以 `√d_k` 相当于方差归一化，让不同 head dimension 下的 logits 尺度稳定。

---

## 拷问 3：softmax 是对哪个维度做？

标准回答：

> 对 key 维度做，也就是 `scores` 的最后一维。对每个 query token，它对所有 key token 的权重和为 1。

---

## 拷问 4：Attention 输出的维度为什么是 `[B,T,d_v]`？

标准回答：

> `softmax(QKᵀ/√d_k)` 的形状是 `[B,T,T]`，表示每个 query 对所有 key/value 的权重；`V` 是 `[B,T,d_v]`。两者相乘后得到 `[B,T,d_v]`，也就是每个 token 汇聚上下文后的新表示。

# 11. 工程排障：模型复读和 Attention 有什么关系？

如果模型突然变成复读机，可能不是 Attention 公式本身错，但可以按这个顺序排查：

```text
1. attention logits 是否异常大
2. softmax 后是否几乎 one-hot
3. scaling 是否写成了 /d_k 或漏掉了 /√d_k
4. dtype 是否导致溢出：fp16 下极端 logits 更容易出问题
5. causal mask 是否写反：是不是把历史遮住、未来放开了
6. RoPE / position_id 是否错位
7. 采样参数是否过于确定：temperature 太低、top_p 太低
8. repetition penalty / stop token / chat template 是否配置错误
```

注意：真实工程中“复读”更多时候来自采样参数、模板、stop token、数据格式或位置编码错位；但如果你自己实现 attention，scaling 和 mask 是第一批必须查的点。

# 12. Day 1 的 LangChain / LangGraph 工程映射

今天的底层主题是 Attention，但路线要求每天都要映射到应用工程。  
Day 1 的映射目标不是做 RAG，而是理解最小 LLM 调用链：

```text
PromptTemplate → ChatModel → OutputParser
```

在 LangChain Expression Language 里常见写法是：

```python
chain = prompt | model | parser
result = chain.invoke({"question": "..."})
```

你可以这样理解：

| 层级 | 负责什么 | 和 Attention 的关系 |
|---|---|---|
| PromptTemplate | 把用户输入组织成模型消息 | 决定模型读到的上下文文本 |
| ChatModel | 真正调用大模型 | 模型内部用 Transformer/Attention 处理 token |
| Runnable / Chain | 把步骤串起来 | 应用编排，不改变模型内部 attention |
| OutputParser | 把模型输出转成结构化结果 | 后处理，不参与模型内部计算 |

今天只要建立边界感：

> LangChain 负责应用编排；Attention 是模型内部的信息读取机制。  
> 你调 PromptTemplate 不会改变 attention 公式，但 prompt 的组织方式会改变模型能关注到的 token 内容。

In [ ]:
# 不依赖 LangChain 的“伪 Runnable”小例子：帮助理解 Prompt → Model → Parser 的工程链路
# 这不是 LLM，只是模拟调用链结构。

class PromptTemplate:
    def __init__(self, template):
        self.template = template

    def invoke(self, variables):
        return self.template.format(**variables)

class FakeChatModel:
    def invoke(self, prompt):
        return f"模型收到的 prompt 是：{prompt}\n\n模拟回答：Attention 的核心是 QKᵀ 计算相关性。"

class StringParser:
    def invoke(self, model_output):
        return model_output.strip()

prompt = PromptTemplate("请用一句话解释：{concept}")
model = FakeChatModel()
parser = StringParser()

step1 = prompt.invoke({"concept": "为什么 QKᵀ 能表示 token 相关性？"})
step2 = model.invoke(step1)
step3 = parser.invoke(step2)

print(step3)

# 13. 今日作业

## 作业 A：手写公式

把下面三行默写到纸上：

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V
$$

$$
S=\frac{QK^\top}{\sqrt{d_k}}
$$

$$
O=\mathrm{softmax}(S)V
$$

---

## 作业 B：改代码

在上面的 NumPy demo 中，修改：

```python
B = 2
T = 8
d_model = 16
d_k = 8
d_v = 8
```

然后重新跑：

1. shape table
2. attention heatmap
3. softmax 饱和图

观察输出维度是否符合预期。

---

## 作业 C：面试口述

用 60 秒回答：

> “请解释 Scaled Dot-Product Attention 的计算过程，以及为什么要除以 √d_k。”

参考结构：

```text
第一句：Attention 的目标是让每个 token 从其他 token 读取上下文信息。
第二句：Q 表示查询意图，K 表示匹配索引，V 表示被读取的内容。
第三句：QKᵀ 得到 [T,T] 的两两匹配分数。
第四句：除以 √d_k 是为了稳定 logits 尺度，避免 softmax 饱和。
第五句：softmax 后得到权重，再乘 V 得到上下文表示。
```

---

## 作业 D：排障题

如果你手写 attention 后发现输出全是 `nan`，优先检查：

```text
1. softmax 前 logits 是否过大
2. 是否做了数值稳定 softmax：x - max(x)
3. mask 是否用了 -inf 后又参与了非法计算
4. dtype 是否溢出
5. sqrt(d_k) 是否写错
```

# 14. 推荐学习资源与观看顺序

建议按这个顺序配合今天的 Notebook 学：

1. **先看可视化直觉**：Jay Alammar - The Illustrated Transformer  
   适合先建立 Q/K/V 和 attention 的图像化理解。  
   https://jalammar.github.io/illustrated-transformer/

2. **再看原论文公式**：Attention Is All You Need  
   重点看 Scaled Dot-Product Attention 一节，确认公式来源。  
   https://arxiv.org/abs/1706.03762

3. **再看代码实现**：Harvard NLP - The Annotated Transformer  
   对照 PyTorch 实现，理解 mask、dropout、matmul 维度。  
   https://nlp.seas.harvard.edu/annotated-transformer/

4. **视频辅助**：3Blue1Brown - Attention in transformers, step-by-step  
   适合理解 attention 在 embedding 空间里“移动信息”的直觉。  
   https://www.3blue1brown.com/lessons/attention

5. **课程辅助**：DeepLearning.AI - Attention in Transformers: Concepts and Code in PyTorch  
   适合系统跟着写 PyTorch 版本。  
   https://www.deeplearning.ai/courses/attention-in-transformers-concepts-and-code-in-pytorch/

6. **系统课程**：Stanford CS224N Self-Attention and Transformers  
   适合面试和理论复盘。  
   https://web.stanford.edu/class/cs224n/

# 15. 今日掌握检查表

运行完 Notebook 后，请逐项打勾：

- [ ] 我能解释 `Q`、`K`、`V` 的角色差异。
- [ ] 我能推导 `QKᵀ` 的形状为什么是 `[B,T,T]`。
- [ ] 我能说明第 `i` 行第 `j` 列的 attention score 含义。
- [ ] 我能从零写出 NumPy 版 scaled dot-product attention。
- [ ] 我能解释为什么不除以 `√d_k` 会让 softmax 饱和。
- [ ] 我能解释 causal mask 为什么用于自回归生成。
- [ ] 我能回答“为什么不是 QQᵀ 或 KKᵀ”。
- [ ] 我知道 LangChain 的 PromptTemplate / ChatModel / Runnable 是应用编排层，不是模型内部 attention。

# 附录：PyTorch 对照实现（可选）

如果你的环境装了 PyTorch，可以运行下面的单元，对照 NumPy 版本。  
没装 PyTorch 也不影响今天主线。

In [ ]:
try:
    import torch

    def torch_scaled_dot_product_attention(Q, K, V):
        d_k = Q.shape[-1]
        scores = Q @ K.transpose(-2, -1)
        weights = torch.softmax(scores / math.sqrt(d_k), dim=-1)
        return weights @ V, weights

    torch.manual_seed(42)
    Q_t = torch.randn(1, 5, 4)
    K_t = torch.randn(1, 5, 4)
    V_t = torch.randn(1, 5, 6)

    O_t, W_t = torch_scaled_dot_product_attention(Q_t, K_t, V_t)
    print("PyTorch output shape:", tuple(O_t.shape))
    print("PyTorch attention weight row sums:", W_t.sum(dim=-1))
except ImportError:
    print("当前环境没有安装 torch。主线学习不受影响；需要时可执行：pip install torch")